In [2]:
from ask_delphi_api.authentication import AskDelphiClient
client = AskDelphiClient()
client.authenticate()

Parsed tenant/project/acl from CMS URL
Loaded cached tokens.
AUTHENTICATION STARTED
Trying cached tokens...
Editing API token status: 200
Editing API token acquired.
SUCCESS using cached tokens!


True

In dit notebook wordt uitgewerkt hoe je topics (op basis van hun timestamp) kunt verwijderen.
Stappenplan
- Ophalen van alle info van alle bestaande topics
- topics filteren op basis van LastModificationDate
- topics die voldoen uitchecken en meteen verwijderen

In [6]:
from ask_delphi_api.project import Project
from ask_delphi_api.topictools import TopicTools


project = Project(client)
topics = TopicTools(client, project)




In [ ]:
all_topics = topics.fetch_topiclist()

In [6]:
import pprint
pprint.pprint(all_topics[1001])

# "b0e911e4-3d0c-432b-bb6c-d06ac939a037"

{'canBreakLocks': True,
 'canEditTopic': True,
 'checkedOutByMe': False,
 'currentUserHasCheckedOutVersions': False,
 'isEditable': True,
 'isLocked': False,
 'lastModificationDate': '2026-02-25T17:46:27.09',
 'lockedByUserId': None,
 'lockedByUserName': None,
 'majorVersionNo': 0,
 'minorVersionNo': 2,
 'previewAvailable': True,
 'publicationRoute': 'https://digitalecoach.askdelphi.com/default/b73bdf7d-7698-4413-8af6-1ede691113df/topic/8b11b869-dc22-4c31-852c-b4dc74178beb',
 'publicationState': 'Processed',
 'thumbnailImageBase64': '',
 'title': 'Toets aan criteria art 19a Uitvoeringsregeling',
 'topicGuid': '8b11b869-dc22-4c31-852c-b4dc74178beb',
 'topicTypeKey': 'c568af9a-6c89-45cf-a580-bc94e1c62ae3',
 'topicTypeName': 'Stap',
 'topicTypeNamespace': 'http://tempuri.org/imola-action',
 'topicVersionKey': '46429d51-6fca-4617-b3e1-3212a348298d'}


In [8]:

def get_workflowstate(topicId: str):  
    endpoint = f"/v3/tenant/{{tenantId}}/project/{{projectId}}/acl/{{aclEntryId}}/topic/{topicId}/workflowstate"
    data = {}
    result = client._request("GET", endpoint, json_data=data)
    return result

pprint.pprint(get_workflowstate("b0e911e4-3d0c-432b-bb6c-d06ac939a037"))

{'canEditErrorCode': None,
 'data': {'canBreakLocks': True,
          'canThisSessionAccessThisTopicForUpdate': True,
          'isDeleted': True,
          'isExisting': True,
          'isExternal': False,
          'lockedBy': '',
          'state': 1,
          'topicTitle': 'Externe URL download'}}


In [ ]:
results = topics.filter_between("2026-01-13T00:00:00", "2026-02-18T23:59:59")
print("Aantal topics binnen de range:", len(results))

In [ ]:
topicIdverwijderen = [r["topicGuid"] for r in results]

for topic_id in topicIdverwijderen:
    try: 
        version_id = topics.get_topicVersionId(topic_id)
        result = topics.delete_topic(topic_id, version_id)
    except Exception as e:
        print(f"FOUT bij verwijderen van {topic_id}: {e}")



In [ ]:
from ask_delphi_api.workflow import Workflow
from ask_delphi_api.import_digicoach import Import
import_service = Import()

def delete_topic(topicId: str, topicVersionId: str, request_id: str):
    """Verwijder een topic."""
    endpoint = f"v3/tenant/{{tenantId}}/project/{{projectId}}/acl/{{aclEntryId}}/topic/{topicId}/topicVersion/{topicVersionId}"
    data = {
        "workflowActions": {
            "applyWorkflowStageIds": [
            request_id
            ],
            "applyWorkflowTransitionIds": [
             request_id            ],
            "increaseMajorVersionNo": True
        }
    }
    return client._request("DELETE", endpoint, json_data=data)

# topic_id = "56365908-5ffb-4c78-a2f5-bd4aea57fc66"
topic_id = "3f65901e-001f-41d9-b8ed-e715876877c3"
workflow = Workflow(client)
request_id = workflow.create_workflow_transition_request(topic_id)

# response = topics.checkout(topic_id)
try:
    topic_version_id = topics.get_topicVersionId(topic_id)
    response = delete_topic(topic_id, topic_version_id, request_id) 
except:
    print("Oeps markeren opruimen lijkt mislukt")

response = topics.checkin(topic_id)
import_service.publiceer(topic_id)


Parsed tenant/project/acl from CMS URL
Loaded cached tokens.
AUTHENTICATION STARTED
Trying cached tokens...
Editing API token status: 200
Editing API token acquired.
SUCCESS using cached tokens!
Body: {"success":false,"errorCode":"ERR40003","errorMessage":"Failed to mark topic (56365908-5ffb-4c78-a2f5-bd4aea57fc66) as deleted. Failed to check-in topic.","response":null}
Oeps markeren opruimen lijkt mislukt
2026-02-26T12:52:48Z
